# Full Report Data POC

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [46]:
import os
import datetime

from dotenv import load_dotenv

import common

## Environmental Variables

In [7]:
load_dotenv()
s3_bucket = os.environ.get("S3_BUCKET")
print(f"S3 Bucket: {s3_bucket}")

S3 Bucket: kasbench-test-20260528-377288663341-us-east-1-an


## Constants

In [57]:
RUN = "exp-2026-09-04-a"
SAMPLE_TRIAL = "trial0001" # Used to extract common infrastructure details
DATA_DIR = f"../data/{RUN}"
os.makedirs(DATA_DIR, exist_ok=True)
# RUN_DB = f"{DATA_DIR}/logs.db"
# print(f"Run database is {RUN_DB}")
report_date = datetime.datetime.now(datetime.UTC)

## Benchmark Metadata Section

### Experiment Progress (experiment_progress.json)

In [ ]:
experiment_progress = common.get_experiment_progress(s3_bucket, RUN)
experiment_progress

{'version': 1,
 'parameters': {'run_identifier': 'exp-2026-09-04-a',
  'trial_prefix': 'trial',
  'autoscalers': ['hpa', 'vpa', 'keda', 'none'],
  'trials_per_autoscaler': 4,
  'run_duration': 30,
  'working_directory': '/home/ubuntu/data/benchmarks',
  's3_bucket': 'kasbench-test-20260528-377288663341-us-east-1-an',
  'aws_region': 'us-east-1',
  'var_files': ['medium.tfvars'],
  'variables': ['spot=false'],
  'auto_approve': True,
  'runner_version': 'latest',
  'health_timeout': 6000,
  'rollout_timeout': 6000,
  'cluster_cidr_range': '10.244.0.0/16',
  'role_params': {'back-office': {'baseLoadIntensity': 500,
    'baseDelayPercentage': 25,
    'spawnRate': 10},
   'portfolio-manager': {'baseLoadIntensity': 500,
    'baseDelayPercentage': 25,
    'spawnRate': 10},
   'trader': {'baseLoadIntensity': 500,
    'baseDelayPercentage': 25,
    'spawnRate': 10},
   'investor': {'baseLoadIntensity': 500,
    'baseDelayPercentage': 25,
    'spawnRate': 10}},
  'ebs_wait': 120},
 'effective_s

### Benchmark Database (benchmark.db)

In [ ]:
benchmark_db_path =f"{DATA_DIR}/benchmark.db"
common.download_benchmark_db(s3_bucket, RUN, benchmark_db_path)
start_time, end_time = common.get_benchmark_start_end(benchmark_db_path)
# print start_time and end_time as formatted strings with timezone
print(f"Start Time: {start_time.strftime('%Y-%m-%d %H:%M:%S %Z')}UTC")
print(f"End Time: {end_time.strftime('%Y-%m-%d %H:%M:%S %Z')}UTC")



Start Time: 2026-09-04 15:26:11 UTC
End Time: 2026-09-05 05:04:00 UTC


### Tofu Outputs (tofu_outputs.json)

In [58]:
tofu_outputs = common.get_tofu_outputs(s3_bucket, RUN, SAMPLE_TRIAL)
tofu_outputs

{'ami_ids': {'sensitive': False,
  'type': ['object',
   {'amd64': 'string', 'ami_runner_amd64': 'string', 'arm64': 'string'}],
  'value': {'amd64': 'ami-0244585558b2aebd7',
   'ami_runner_amd64': 'ami-03a891c9de365d954',
   'arm64': 'ami-052bbac83b5bb0ab9'}},
 'benchmark_runner': {'sensitive': False,
  'type': ['object',
   {'ami_id': 'string',
    'architecture': 'string',
    'instance_id': 'string',
    'instance_type': 'string',
    'private_ip': 'string',
    'public_ip': 'string',
    'root_volume_id': 'string',
    'subnet_id': 'string'}],
  'value': {'ami_id': 'ami-03a891c9de365d954',
   'architecture': 'amd64',
   'instance_id': 'i-0525cc5a126382670',
   'instance_type': 'c6a.large',
   'private_ip': '10.0.1.61',
   'public_ip': '34.239.121.74',
   'root_volume_id': 'vol-09705991506cd119b',
   'subnet_id': 'subnet-046946fc478968053'}},
 'control_plane': {'sensitive': False,
  'type': ['object',
   {'ami_id': 'string',
    'architecture': 'string',
    'instance_id': 'string',

### Run Details (run_details.json)

In [71]:
run_details = common.get_run_details(s3_bucket, RUN, SAMPLE_TRIAL)
run_details

{'timestamp': '2026-09-04T16:05:20.717849+00:00',
 'environment': {'HOST': '0.0.0.0',
  'PORT': 8080,
  'SSH_USER': 'ubuntu',
  'SSH_CONNECT_TIMEOUT': 30,
  'NODE_READINESS_TIMEOUT_SECONDS': 300,
  'NODE_READINESS_POLL_INTERVAL': 10,
  'HEALTH_CHECK_MAX_ATTEMPTS': 3,
  'HEALTH_CHECK_INTERVAL_SECONDS': 5,
  'RABBITMQ_IMAGE': 'rabbitmq:4-management',
  'HTTP_CONNECT_TIMEOUT': 10,
  'HTTP_READ_TIMEOUT': 30,
  'MANIFEST_FETCH_TIMEOUT': 30},
 'initialization': {'autoscaler': 'none',
  'controlPlaneNode': '10.0.1.94',
  'amdWorkerNodes': ['10.0.1.14'],
  'armWorkerNodes': ['10.0.1.40'],
  's3Bucket': 'kasbench-test-20260528-377288663341-us-east-1-an',
  'globecoUrl': 'http://kasb-20260904152134310200000012-c74e81f985d05edf.elb.us-east-1.amazonaws.com',
  'runIdentifier': 'exp-2026-09-04-a',
  'trialIdentifier': 'trial0001',
  'clusterCidrRange': '10.244.0.0/16',
  'kubernetesVersion': '1.36.1',
  'loadGeneratorImage': 'kasbench/kasbench-load-generator:latest',
  'runDurationMinutes': 30,
  '

In [76]:
metadata = {}
metadata["benchmark_name"] = {"name": "Benchmark Name", "value": "KASBench"}
metadata["version"] = {"name": "Version", "value": experiment_progress["version"]} 
metadata["testing_dates"] = {"name": "Testing Dates", "value": f"{start_time.strftime('%Y-%m-%d %H:%M:%S')} UTC to {end_time.strftime('%Y-%m-%d %H:%M:%S')} UTC"}
metadata["reporting_date"] = {"name": "Reporting Date", "value": report_date.strftime('%Y-%m-%d %H:%M:%S') + " UTC"}
metadata["benchmark_application"] = {"name": "Benchmark Application", "value": "GlobeCo"}
metadata["cloud_provider"] = {"name": "Cloud Provider", "value": "Amazon Web Services"}
metadata["region"] = {"name": "Region", "value": experiment_progress["parameters"]["aws_region"]}
metadata["bencmark_runner"] = {"name": "Benchmark Runner", 
    "value": tofu_outputs["benchmark_runner"]["value"]["instance_type"] + " (" + 
    tofu_outputs["benchmark_runner"]["value"]["architecture"] + ")"}
metadata["control_plane"] = {"name": "Control Plane", 
    "value": tofu_outputs["control_plane"]["value"]["instance_type"] + " (" + 
    tofu_outputs["control_plane"]["value"]["architecture"] + ")"}

worker_nodes = tofu_outputs["worker_nodes"]["value"]
amd_nodes = len(worker_nodes["amd64"])
arm_nodes = len(worker_nodes["arm64"])
if amd_nodes:
    amd_instance_type = worker_nodes["amd64"][0]["instance_type"]
    metadata["worker_nodes_amd"] = {"name": "Worker Nodes AMD", "value": f"{amd_nodes} x {amd_instance_type}"}
if arm_nodes:
        arm_instance_type = worker_nodes["arm64"][0]["instance_type"]
        metadata["worker_nodes_arm"] = {"name": "Worker Nodes ARM", "value": f"{arm_nodes} x {arm_instance_type}"}

metadata["storage_type_ios"] = {"name": "Storage Type", "value": "EBS gp3 3000 IOPS 125 MiB/s"}

ami_ids = tofu_outputs["ami_ids"]["value"]

metadata["ami_amd64"] = ami_ids["amd64"]
metadata["ami_arm64"] = ami_ids["arm64"]
metadata["ami_runner_amd64"] = ami_ids["ami_runner_amd64"]

kubernetes_version = run_details["initialization"]["kubernetesVersion"]
metadata["kubernetes_version"] = {"name": "Kubernetes Version", "value": kubernetes_version}

metadata["Container Runtime"] = "Containerd"

autoscalers = experiment_progress["parameters"]["autoscalers"]
autoscalers = [a.upper() for a in autoscalers]

metadata["autoscalers"] = {"name": "Autoscalers Evaluated", "value": ", ".join(autoscalers)}

trials_per_autoscaler = experiment_progress["parameters"]["trials_per_autoscaler"]

metadata["trials_per_autoscaler"] = {"name": "Trials Per Autoscaler", "value": trials_per_autoscaler}

metadata

{'benchmark_name': {'name': 'Benchmark Name', 'value': 'KASBench'},
 'version': {'name': 'Version', 'value': 1},
 'testing_dates': {'name': 'Testing Dates',
  'value': '2026-09-04 15:26:11 UTC to 2026-09-05 05:04:00 UTC'},
 'reporting_date': {'name': 'Reporting Date',
  'value': '2026-09-07 23:58:54 UTC'},
 'benchmark_application': {'name': 'Benchmark Application',
  'value': 'GlobeCo'},
 'cloud_provider': {'name': 'Cloud Provider', 'value': 'Amazon Web Services'},
 'region': {'name': 'Region', 'value': 'us-east-1'},
 'bencmark_runner': {'name': 'Benchmark Runner', 'value': 'c6a.large (amd64)'},
 'control_plane': {'name': 'Control Plane', 'value': 'm8i.large (amd64)'},
 'worker_nodes_amd': {'name': 'Worker Nodes AMD', 'value': '1 x c6a.2xlarge'},
 'worker_nodes_arm': {'name': 'Worker Nodes ARM', 'value': '1 x c6g.2xlarge'},
 'storage_type_ios': {'name': 'Storage Type',
  'value': 'EBS gp3 3000 IOPS 125 MiB/s'},
 'ami_amd64': 'ami-0244585558b2aebd7',
 'ami_arm64': 'ami-052bbac83b5bb0ab9